In [1]:
#! mlflow server -p 5001

In [ ]:
import mlflow
from openai import OpenAI
from loguru import logger
from mlflow.entities import SpanType
import time

/opt/homebrew/Caskroom/miniconda/base/envs/agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
PORT = "5001"
EXPERIMENT_NAME = "Chatbot"
TRACE_ID = "id-12345"
TRACE_KEY = "chatbot_version"
TRACE_VALUE = "1.0"

OLLAMA_API_URL = "http://localhost:11434/v1"
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [ ]:
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774097476777, experiment_id='1', last_update_time=1774097476777, lifecycle_stage='active', name='Chatbot', tags={}, workspace='default'>

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.client = self._set_llm(OLLAMA_API_URL)
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})
            logger.info(f"System message set: {system}")

    def _set_llm(self, url):
        try:
            logger.info(f"Ollama API URL: {url}")
            return OpenAI(base_url=url, 
                          api_key="dummy")
        except Exception as e:
            print(f"Error initializing OpenAI client: {e}")
            return None

    @mlflow.trace(span_type=SpanType.CHAIN)
    def __call__(self, user_input):
        try:
            self.messages.append({"role": "user", "content": user_input})
            logger.info(f"User message added: {user_input}")
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result})
            logger.info(f"Assistant response added: {result}")
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        try:
            response = self.client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=self.messages,
                        temperature=TEMPERATURE,
                        max_tokens=MAX_TOKENS,
                    )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Error during agent execution: {e}")
            return "An error occurred while processing your request."

## AI Chatbot

In [ ]:
def start_session():
    agent = Agent(system="You are a helpful assistant.")
    while True:
        user_query = input("You: ")
        if user_query == "BYE":
            break
        response = agent(user_query)

start_session()

2026-03-21 14:06:22.786 | INFO     | __main__:_set_llm:14 - Ollama API URL: http://localhost:11434/v1
2026-03-21 14:06:22.806 | INFO     | __main__:__init__:10 - System message set: You are a helpful assistant.
2026-03-21 14:06:26.079 | INFO     | __main__:__call__:25 - User message added: ciao
2026-03-21 14:06:28.571 | INFO     | __main__:__call__:28 - Assistant response added: Ciao! Come posso aiutarti oggi?
2026-03-21 14:06:36.315 | INFO     | __main__:__call__:25 - User message added: si parlami di te?
2026-03-21 14:06:41.322 | INFO     | __main__:__call__:28 - Assistant response added: Certo! Sono ChatGPT, un modello di intelligenza artificiale sviluppato da OpenAI. Ecco un po’ di “biografia” su di me:

| **Cosa sono
2026-03-21 14:06:47.608 | INFO     | __main__:__call__:25 - User message added: non è chiaro
2026-03-21 14:06:52.745 | INFO     | __main__:__call__:28 - Assistant response added: Mi scuso se la risposta


[Trace(trace_id=tr-72d88166f7340335cb91ffabdbf45e53), Trace(trace_id=tr-3cbd7243649ee540f3d39ad957c8812f), Trace(trace_id=tr-d9de6ef7747371812d4476a71e94b79b)]